Zuerst Vektoren laden, dann werden jeweils die Schnittmengen von dem Base-Jahr und dem zu rotierenden Jahr in einer Pickele datei abgespeichert

In [2]:
import sys
import pandas as pd
from pathlib import Path
import pickle
import gensim
from gensim.models import KeyedVectors
from tqdm import tqdm
import os
import shutil
sys.path.append(os.path.abspath("."))
from CREATE_DICT_MOST_SIMILAR_MODERN import run_most_similar
from ALIGNMENT_MODERN import run_sense_alignment

v_path_base = Path("Data/Vektoren")
v_path_16 = v_path_base / "2016" / "vektoren_2016.txt"
v_path_17 = v_path_base / "2017" / "vektoren_2017.txt"
output_dir = Path("SeNSe-main/SeNSe-main/data/")


In [16]:
vectors_2016 = pd.read_csv(v_path_16, sep=" ", header=None, skiprows=1)
vectors_2017 = pd.read_csv(v_path_17, sep=" ", header=None, skiprows=1)
print(len(vectors_2016))
len((vectors_2017))

16618


18426

In [18]:
s_names_16 = set(vectors_2016.iloc[:,0])
s_names_17 = set(vectors_2017.iloc[:,0])
print(list(s_names_16)[:5])
print(list(s_names_17)[:5])
print(len(list(s_names_16)))
print(len(list(s_names_17)))

common_subs = s_names_16 & s_names_17
print(len(list(common_subs)))

['youseeingthisshit', 'blacklagoon', 'ssbbw', 'Silverbugs', 'progressive']
['youseeingthisshit', 'blacklagoon', 'ssbbw', 'Silverbugs', 'progressive']
16618
18426
16439


Schnittmenge als Pickle Datei speichern für SeNse

In [24]:
common_subs_dict = {s: s for s in common_subs}
with open("SeNSe-main/SeNSe-main/data/translation_dictionary_16_17.pkl", "wb") as f:
    pickle.dump(common_subs_dict,f)




Dictionary in beide Richtungen speichern

In [33]:


 # 1. Wo liegt dein Original?
pfad_original = "SeNSe-main/SeNSe-main/data/translation_dictionary_16_17.pkl"

# 2. Wie muss die Kopie heißen?
pfad_kopie = "SeNSe-main/SeNSe-main/data/translation_dictionary_17_16.pkl"

# 3. Datei kopieren
shutil.copy(pfad_original, pfad_kopie)
print(f"Kopie erstellt: {pfad_kopie}")

Kopie erstellt: SeNSe-main/SeNSe-main/data/translation_dictionary_17_16.pkl


Prüfen

In [25]:
import pickle
file_path = "SeNSe-main/SeNSe-main/data/translation_dictionary_16_17.pkl"

with open(file_path, "rb") as f:
     check_dict = pickle.load(f)

print(f"Anzahl der Einträge: {len(check_dict)}")
print(f"Typ: {type(check_dict)}")
print("Beispiele:")

for i, (k, v) in enumerate(list(check_dict.items())[:5]):
    print(f"{k}: {v}")

Anzahl der Einträge: 16439
Typ: <class 'dict'>
Beispiele:
youseeingthisshit: youseeingthisshit
blacklagoon: blacklagoon
ssbbw: ssbbw
Silverbugs: Silverbugs
progressive: progressive


Nun die optimalen Anker berechnen

In [30]:
JAHRE_SIM = ["2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"]

for jahr in JAHRE_SIM:
    run_most_similar(
    src_lang=jahr[2:], # '2018' -> '18'
    trg_lang='16',
    src_path=f'Data/Vektoren/{jahr}/vektoren_{jahr}.txt',
    trg_path='Data/Vektoren/2016/vektoren_2016.txt'
    )

Starting process for years 16 and 17...
Loading source model...
Loading target model...
Computing Top-200 neighbors for 16...


100%|██████████| 16618/16618 [00:16<00:00, 1033.34it/s]


Saving SeNSe-main/SeNSe-main/data/most_similar_dictionary_16.pkl...
Computing Top-200 neighbors for 17...


100%|██████████| 18426/18426 [00:24<00:00, 752.40it/s] 


Saving SeNSe-main/SeNSe-main/data/most_similar_dictionary_17.pkl...
Total elapsed time: 51.54 seconds


Alignment

In [6]:


import sys
import os
sys.path.append(os.path.abspath("."))

# Importiere die FULL Version
from ALIGNMENT_FULL_SENSE import run_full_sense_alignment


### Master-Loop: Automatisierung für alle Jahre (2017-2024)
Dieser Loop führt alle Schritte (Schnittmenge, Nachbarschaften, Alignment) in einem Rutsch für alle Jahre durch.

In [7]:
# --- MASTER-LOOP FÜR ALLE JAHRE ---

# Liste der Jahre, die du verarbeiten willst
JAHRE = ["2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"]
data_dir = "SeNSe-main/SeNSe-main/data"

# 1. Basis-Subreddits (2016) einmalig laden (für die Schnittmengen)
print("Lade Basis-Subreddits (2016) für Schnittmengen-Berechnung...")
df_16_names = pd.read_csv("Data/Vektoren/vektoren_2016.txt", sep=" ", header=None, skiprows=1, usecols=[0])
s_names_16 = set(df_16_names.iloc[:, 0])

# 2. Der Loop über alle Jahre
for jahr in JAHRE:
    print(f"\n{'='*60}")
    print(f"STARTE VERARBEITUNG FÜR JAHR: {jahr}")
    print(f"{'='*60}")
    
    label = jahr[2:] # Macht aus '2017' -> '17'
    path_vec_jahr = f"Data/Vektoren/vektoren_{jahr}.txt"
    path_vec_16 = "Data/Vektoren/vektoren_2016.txt"
    
    # --- SCHRITT A: Translation-Dictionary (Schnittmenge) ---
    print(f"Erstelle Translation-Dictionary für 16 <-> {label}...")
    df_jahr_names = pd.read_csv(path_vec_jahr, sep=" ", header=None, skiprows=1, usecols=[0])
    s_names_jahr = set(df_jahr_names.iloc[:, 0])
    
    common = s_names_16 & s_names_jahr
    common_dict = {s: s for s in common}
    
    # Speichern in beide Richtungen (wichtig für SeNSe)
    for suffix in [f"16_{label}", f"{label}_16"]:
        with open(os.path.join(data_dir, f"translation_dictionary_{suffix}.pkl"), "wb") as f:
            pickle.dump(common_dict, f)

    # --- SCHRITT B: Nachbarschaften (Most Similar) ---
    # Hinweis: Dies ist nur nötig, wenn run_most_similar noch nicht für dieses Jahr lief
    run_most_similar(
        src_lang=label,
        trg_lang='16',
        src_path=path_vec_jahr,
        trg_path=path_vec_16,
        output_dir=data_dir
    )

    # --- SCHRITT C: Full SeNSe Alignment ---
    print(f"Führe FULL SeNSe Alignment durch (Toleranz 0.8)...")
    # Wir setzen perform_alignment=True, um die rotierten Vektoren zu erzeugen
    res_df, out_path = run_full_sense_alignment(
        lang_src=label,
        lang_trg='16',
        path_vec_src=path_vec_jahr,
        path_vec_trg=path_vec_16,
        tolerance_limit=0.8,
        perform_alignment=True
    )
    
    # Anker-Liste zur Dokumentation speichern
    res_df.to_csv(f"final_anker_16_{jahr}_full.csv", index=False)
    print(f"FERTIG: Jahr {jahr} erfolgreich ausgerichtet.")
    print(f"Ergebnis-Vektoren unter: {out_path}")

print("\n" + "#"*60)
print("GLÜCKWUNSCH! ALLE JAHRE WURDEN ERFOLGREICH AUF 2016 AUSGERICHTET.")
print("#"*60)

Lade Basis-Subreddits (2016) für Schnittmengen-Berechnung...

STARTE VERARBEITUNG FÜR JAHR: 2017
Erstelle Translation-Dictionary für 16 <-> 17...
Starting process for labels: 17 and 16
Source Path: Data/Vektoren/vektoren_2017.txt
Target Path: Data/Vektoren/vektoren_2016.txt
Loading source model...
Loading target model...
Computing Top-200 neighbors for 17...


100%|██████████| 18426/18426 [00:47<00:00, 384.28it/s]


Saving SeNSe-main/SeNSe-main/data/most_similar_dictionary_17.pkl...
Computing Top-200 neighbors for 16...


100%|██████████| 16618/16618 [00:43<00:00, 378.31it/s] 


Saving SeNSe-main/SeNSe-main/data/most_similar_dictionary_16.pkl...
Total elapsed time: 104.38 seconds
Führe FULL SeNSe Alignment durch (Toleranz 0.8)...
Step 0: Loading models and dictionaries...
Step 1 & 2: Computing SNDCG and Selecting Best Anchors...


...

Saving SeNSe-main/SeNSe-main/data/most_similar_dictionary_16.pkl...
Total elapsed time: 75.73 seconds
Führe FULL SeNSe Alignment durch (Toleranz 0.8)...
Step 0: Loading models and dictionaries...
Step 1 & 2: Computing SNDCG and Selecting Best Anchors...


100%|██████████| 15893/15893 [00:03<00:00, 4101.66it/s]


Anchors after tolerance filter: 3177
Step 3: Dispersion (Removing spatially clustered anchors)...


100%|██████████| 3177/3177 [00:00<00:00, 1036343.43it/s]


Anchors after Dispersion: 2252
Step 4 & 5: Performing Procrustes Alignment...
Alignment complete! Saved to: SeNSe-main/SeNSe-main/output/projected_24_onto_16_FULL.txt
FERTIG: Jahr 2024 erfolgreich ausgerichtet.
Ergebnis-Vektoren unter: SeNSe-main/SeNSe-main/output/projected_24_onto_16_FULL.txt

############################################################
GLÜCKWUNSCH! ALLE JAHRE WURDEN ERFOLGREICH AUF 2016 AUSGERICHTET.
############################################################
